# Experiment 2A · Readmission explanation infrastructure

H1 compares attribution faithfulness between A (baseline) and B (SHAP-grounded).
H2 compares B's original draft with C (verification, optional revision, and fallback).
This notebook exercises the workflow on fabricated cases with known attribution references.

## Environment and configuration

Select the HRP Python kernel created by `uv sync --locked`. The default
workflow uses scripted responses. `Ollama.discover()` prepares the local
Qwen3 adapter for a separate fictional smoke run.

In [1]:
import json
from pathlib import Path
import sys
import tempfile

from IPython.display import display

source = next(path for path in (Path("src"), Path("."), Path("HRP/src"))
              if (path / "llm_runner.py").is_file())
sys.path.insert(0, str(source.resolve()))

from experiment2_smoke import fictional_cases, run_smoke
from llm_runner import ModelConfig, Ollama, run_experiment
from llm_verification import RESPONSE_SCHEMA, build_prompt

ModelConfig()

/Users/sghosh/Documents/Github/AI380H/HRP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModelConfig(model='qwen3:8b', endpoint='http://127.0.0.1:11434', digest=None, temperature=0, seed=42, num_ctx=4096, num_predict=512)

## Explanation schema

All conditions return an explanation and up to three ordered drivers.
Driver directions are `positive` or `negative`. Sentence-count
violations are recorded in the parse trace.

In [2]:
display(RESPONSE_SCHEMA)

{'explanation': 'Nonempty string, at most 90 words. Aim for 2–3 sentences.',
 'drivers': [{'feature': 'supplied feature key',
   'direction': 'positive or negative'}]}

## Prompt templates

A sees the same feature values, units, and probability as B.
B additionally sees the SHAP reference. C begins from B's raw text.

In [3]:
example = fictional_cases()[0]
for condition in ("A", "B"):
    print(f"Condition {condition}\n{build_prompt(example, condition)}\n")

Condition A
Explain which available features most strongly influence the fitted model's readmission prediction.
State the supplied predicted probability as a percentage. Use 2–3 sentences and at most 90 words.
Describe model behavior. Avoid clinical diagnosis, causal claims, treatment advice, and invented measurements.
Use explicit feature names and direct statements about increases or decreases in the model's predicted risk.
You may express uncertainty and list fewer drivers, including none when the evidence is insufficient.
Return only a JSON object with exactly two keys: "explanation" (a string) and "drivers" (an array of 0–3 objects).
Each driver must have exactly "feature" (a supplied feature key) and "direction" ("positive" or "negative").
List drivers from most to least influential and use each feature at most once.
Attributions are unavailable here. Acknowledge uncertainty when you cannot infer model behavior.

Input JSON:
{"features": [{"name": "age", "label": "Age", "unit": "

## Fictional smoke-test setup

The fixture generator checks additivity against an independent linear
margin. Scripted responses exercise ranked attributions, abstention,
malformed JSON, a missing lab, contradictory prose, and revision failure.

In [4]:
output_directory = Path(tempfile.mkdtemp(prefix="hrp-experiment2-fictional-"))

## Runner invocation

The smoke workflow checkpoints at a four-call budget, then resumes
and verifies that a completed rerun uses zero additional calls.

In [5]:
report = run_smoke(output_directory)
display(report)

{'cases': 8,
 'outputs': {'A': 8, 'B': 8, 'C': 8},
 'valid_outputs': {'A': 8, 'B': 7, 'C': 8},
 'calls': 21,
 'additional_calls_on_completed_resume': 0,
 'total_call_latency_seconds': 0.00044725104817189276,
 'accepted_originals': 3,
 'revisions_requested': 5,
 'revisions_accepted': 2,
 'fallbacks': 3,
 'parser_failures': 1,
 'narrative_failures': 6,
 'inference_failures': 1,
 'checkpoint': '/private/var/folders/sb/xd5lskw56px2y86njz2clqpr0000gp/T/hrp-experiment2-fictional-3je_37fx/checkpoint.json'}

## Verification trace

Inspect one failed revision and its deterministic output.

In [6]:
checkpoint = json.loads(Path(report["checkpoint"]).read_text())
records = list(checkpoint["records"].values())
fallback = next(row["verification"] for row in records
                if row["condition"] == "C" and row["verification"]["fallback_used"])
display(fallback)

{'original_raw': '{"explanation": "Sodium was 138 mEq/L. Sodium causes readmission.", "drivers": [{"feature": "sodium_missing", "direction": "negative"}, {"feature": "age", "direction": "positive"}, {"feature": "creatinine", "direction": "negative"}]}',
 'original': {'parse': {'raw': '{"explanation": "Sodium was 138 mEq/L. Sodium causes readmission.", "drivers": [{"feature": "sodium_missing", "direction": "negative"}, {"feature": "age", "direction": "positive"}, {"feature": "creatinine", "direction": "negative"}]}',
   'answer': {'explanation': 'Sodium was 138 mEq/L. Sodium causes readmission.',
    'drivers': [{'feature': 'sodium_missing', 'direction': 'negative'},
     {'feature': 'age', 'direction': 'positive'},
     {'feature': 'creatinine', 'direction': 'negative'}]},
   'error': None,
   'warnings': []},
  'structured': {'expected': [{'feature': 'sodium_missing',
     'direction': 'negative'},
    {'feature': 'age', 'direction': 'positive'},
    {'feature': 'creatinine', 'directi

## Readiness summary

Eight fictional cases exercise all three conditions and recovery from
an interrupted budget. The verifier checks a controlled set of prose
patterns and records ambiguous claims for review. The patient-level
benchmark is outside this milestone.